# PaySim Fraud Detection — XGBoost Pipeline

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_curve, roc_auc_score, f1_score
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print('All imports OK')

## Cell 2 — Load & Clean Data

In [ ]:
df = pd.read_csv('paysim.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nClass distribution:')
print(df['isFraud'].value_counts())
print('\nNull values:')
print(df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

## Cell 3 — Drop Useless Columns & Duplicates

In [ ]:
# Drop ID columns and useless flag
df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], inplace=True)

# Drop nulls and duplicates
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

print('Shape after cleaning:', df.shape)

## Cell 4 — Feature Engineering

In [ ]:
# Balance error features — strongest fraud signals in PaySim
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']

# Encode transaction type
df = pd.get_dummies(df, columns=['type'])

print('Features after engineering:', df.columns.tolist())
print('Shape:', df.shape)

## Cell 5 — Train / Test Split

In [ ]:
X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keeps fraud ratio in both sets
)

print(f'Train size : {X_train.shape[0]}')
print(f'Test size  : {X_test.shape[0]}')
print(f'\nTrain fraud count: {y_train.sum()}')
print(f'Test  fraud count: {y_test.sum()}')

## Cell 6 — Undersample Majority (train only)

In [ ]:
fraud_count = y_train.sum()
majority_target = int(fraud_count * 10)   # keep 10x majority vs minority

under = RandomUnderSampler(
    sampling_strategy={0: majority_target},
    random_state=42
)
X_train_under, y_train_under = under.fit_resample(X_train, y_train)

print('After undersampling:')
print(pd.Series(y_train_under).value_counts())

## Cell 7 — SMOTE Minority (on undersampled train)

In [ ]:
smote = SMOTE(
    sampling_strategy=0.5,   # minority = 50% of majority
    random_state=42,
    k_neighbors=5
)
X_train_bal, y_train_bal = smote.fit_resample(X_train_under, y_train_under)

print('After SMOTE:')
print(pd.Series(y_train_bal).value_counts())

## Cell 8 — Scale Features

In [ ]:
scaler = StandardScaler()

X_train_bal = scaler.fit_transform(X_train_bal)   # fit + transform on train
X_test_scaled = scaler.transform(X_test)           # transform only on test

print('Scaling done')
print('X_train_bal shape:', X_train_bal.shape)
print('X_test_scaled shape:', X_test_scaled.shape)

## Cell 9 — Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2, 0.5],
}

base_model = XGBClassifier(
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring='f1',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_bal, y_train_bal)

print('Best params:', search.best_params_)
print(f'Best CV F1 : {search.best_score_:.4f}')

## Cell 10 — Train Final Model

In [ ]:
best_model = XGBClassifier(
    **search.best_params_,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

best_model.fit(
    X_train_bal, y_train_bal,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

print('Training complete')

## Cell 11 — Find Optimal Threshold

In [ ]:
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f'Optimal threshold: {best_threshold:.4f}')

# Plot PR curve
plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label='PR Curve')
plt.axvline(recall[np.argmax(f1_scores)], color='red', linestyle='--', label=f'Best threshold = {best_threshold:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.show()

## Cell 12 — Evaluate

In [ ]:
y_pred = (y_proba >= best_threshold).astype(int)

print('── Classification Report ─────────────────')
print(classification_report(y_test, y_pred, digits=4))

print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## Cell 13 — Feature Importance

In [ ]:
feature_names = df.drop(columns=['isFraud']).columns
importances = best_model.feature_importances_

feat_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_df, x='importance', y='feature')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

## Cell 14 — Save Model, Scaler & Threshold

In [ ]:
joblib.dump(best_model,     'fraud_model_xgb.pkl')
joblib.dump(scaler,         'scaler.pkl')
joblib.dump(best_threshold, 'threshold.pkl')

print('Saved:')
print('  fraud_model_xgb.pkl')
print('  scaler.pkl')
print('  threshold.pkl')

## Cell 15 — Load & Predict (inference example)

In [ ]:
# Load saved artifacts
model     = joblib.load('fraud_model_xgb.pkl')
scaler    = joblib.load('scaler.pkl')
threshold = joblib.load('threshold.pkl')

# Example: predict on test set
X_new_scaled = scaler.transform(X_test)   # your new data here
proba = model.predict_proba(X_new_scaled)[:, 1]
pred  = (proba >= threshold).astype(int)

print('Predictions:', pred[:10])
print('Probabilities:', proba[:10].round(4))